# Normalización completa del dataset de población de Galicia

## 1. Importar librerías necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from difflib import get_close_matches

## 2. Cargar el dataset de población a normalizar

In [2]:
# Definir la ruta del dataset de población
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\04 - Poblacion\01 - poblacion galicia municipios normalizados.csv'

# Cargar el archivo CSV
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

Filas cargadas: 2921920
Columnas disponibles: ['poblacion', 'densidad_poblacion', 'crecimiento_poblacion', 'superficie_km2', 'fecha', 'municipio', 'latitud', 'longitud']
Tamaño del dataset: 2921920 filas x 8 columnas


,poblacion,densidad_poblacion,crecimiento_poblacion,superficie_km2,fecha,municipio,latitud,longitud
0,8.048130,0.097686,-0.000091,82.388065,1995-01-01,ferrol,43.484571,-8.232997
1,1.516606,0.052624,-0.000015,28.819931,1995-01-01,fisterra,42.906477,-9.263789
2,0.617464,0.015417,-0.000006,40.050962,1995-01-01,paderne,43.296179,-8.156219
3,1.754705,0.035964,-0.000024,48.790271,1995-01-01,padrón,42.739023,-8.660250
4,0.345283,0.002651,-0.000003,130.252600,1995-01-01,o pino,42.945490,-8.350221


In [3]:
# Normalizar la columna de fecha a tipo datetime y dejar solo la fecha (sin hora)
# Ajusta el nombre de la columna si no es exactamente 'fecha'
col_fecha = 'fecha' if 'fecha' in df.columns else df.columns[0]  # Asume que la primera columna es la fecha si no existe 'fecha'
df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce').dt.date
print(f"Columna '{col_fecha}' normalizada. Ejemplo de valores:")
print(df[col_fecha].dropna().astype(str).unique()[:5])

Columna 'fecha' normalizada. Ejemplo de valores:
['1995-01-01' '1995-01-02' '1995-01-03' '1995-01-04' '1995-01-05']


In [4]:
# Filtrar registros solo entre el 1 de enero de 2000 y el 31 de diciembre de 2022
fecha_inicio = pd.to_datetime('2000-01-01').date()
fecha_fin = pd.to_datetime('2022-12-31').date()
df = df[(df[col_fecha] >= fecha_inicio) & (df[col_fecha] <= fecha_fin)]
print(f'Registros tras filtrar por fecha: {len(df)}')

Registros tras filtrar por fecha: 2337600


In [5]:
# Guardar el dataset limpio en la ruta indicada
import os
ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\04 - Poblacion'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - poblacion normalizado completo.csv')
df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset limpio guardado en: {archivo_export}')

Dataset limpio guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\04 - Poblacion\01 - poblacion normalizado completo.csv


### Normalización de tipos tras la exportación del dataset de población
Ejecuta la siguiente celda para revisar y normalizar los tipos de datos del archivo final exportado.

In [6]:
# Cargar y normalizar tipos del dataset final de población
import pandas as pd

archivo_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\04 - Poblacion\01 - poblacion normalizado completo.csv'

# Cargar con low_memory=False para evitar warnings
df_check = pd.read_csv(archivo_export, low_memory=False)

# Convertir fecha
if 'fecha' in df_check.columns:
    df_check['fecha'] = pd.to_datetime(df_check['fecha'], errors='coerce').dt.date

# Convertir a numérico las columnas que deberían serlo (ajusta la lista según tus necesidades)
cols_numericas = [
    'poblacion', 'densidad', 'superficie'
 ]
for col in cols_numericas:
    if col in df_check.columns:
        df_check[col] = pd.to_numeric(df_check[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

# Revisar tipos finales
print('Tipos de datos tras normalización:')
print(df_check.dtypes)
df_check.head()

Tipos de datos tras normalización:
poblacion                float64
densidad_poblacion       float64
crecimiento_poblacion    float64
superficie_km2           float64
fecha                     object
municipio                 object
latitud                  float64
longitud                 float64
dtype: object


,poblacion,densidad_poblacion,crecimiento_poblacion,superficie_km2,fecha,municipio,latitud,longitud
0,7.883628,0.095689,0.000191,82.388065,2000-01-01,ferrol,43.484571,-8.232997
1,1.488961,0.051664,0.000040,28.819931,2000-01-01,fisterra,42.906477,-9.263789
2,0.606814,0.015151,0.000016,40.050962,2000-01-01,paderne,43.296179,-8.156219
3,1.712105,0.035091,0.000044,48.790271,2000-01-01,padrón,42.739023,-8.660250
4,0.339703,0.002608,0.000009,130.252600,2000-01-01,o pino,42.945490,-8.350221


In [7]:
# Guardar el DataFrame normalizado de población con tipos corregidos en un nuevo archivo
ruta_export_final = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\04 - Poblacion'
archivo_export_final = ruta_export_final + '\\02 - poblacion normalizado completo final.csv'
df_check.to_csv(archivo_export_final, index=False, encoding='utf-8')
print(f'Dataset final normalizado guardado en: {archivo_export_final}')

Dataset final normalizado guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\04 - Poblacion\02 - poblacion normalizado completo final.csv
